In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
from scipy.signal import butter, filtfilt
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import seaborn as sns

In [ ]:
def whale_denoise(y, sr, low_freq=20, high_freq=1000):
    """
    Специализированное шумоподавление для низкочастотных китовых звуков
    """
    try:
        # 1. Bandpass фильтр для частот китов
        def butter_bandpass(lowcut, highcut, fs, order=4):
            nyq = 0.5 * fs
            low = lowcut / nyq
            high = highcut / nyq
            b, a = butter(order, [low, high], btype='band')
            return b, a
        
        b, a = butter_bandpass(low_freq, high_freq, sr)
        y_filtered = filtfilt(b, a, y)
        
        # 2. Spectral subtraction
        D = librosa.stft(y_filtered, n_fft=2048, hop_length=512)
        magnitude = np.abs(D)
        phase = np.angle(D)
        
        # Оценка шума по первым 0.2 секундам
        noise_frames = magnitude[:, :int(0.2 * sr / 512)]
        noise_profile = np.median(noise_frames, axis=1)
        
        # Мягкое spectral subtraction
        threshold = 1.5 * noise_profile.reshape(-1, 1)
        magnitude_clean = np.maximum(magnitude - threshold, 0.001)
        
        # Восстановление сигнала
        D_clean = magnitude_clean * np.exp(1j * phase)
        y_clean = librosa.istft(D_clean)
        
        return y_clean
        
    except Exception as e:
        print(f"Ошибка в шумоподавлении: {e}")
        return y

In [ ]:

def compare_spectrograms(audio_file, label, fmax=2000, sr=22050):
    """
    Сравнение спектрограмм до и после шумоподавления
    """
    try:
        y_orig, sr = librosa.load(audio_file, sr=sr, duration=2.0)
        y_clean = whale_denoise(y_orig, sr)
        filename = audio_file.stem
        
        fig, axes = plt.subplots(2, 3, figsize=(18, 8))
        
        # 1. Mel-spectrogram
        spec_orig = librosa.feature.melspectrogram(y=y_orig, sr=sr, n_mels=128, fmax=fmax)
        spec_orig_db = librosa.power_to_db(spec_orig, ref=np.max)
        librosa.display.specshow(spec_orig_db, sr=sr, x_axis='time', y_axis='mel', 
                               ax=axes[0, 0], cmap='viridis', fmax=fmax)
        axes[0, 0].set_title(f'Mel-spectrogram\nДО')
        
        spec_clean = librosa.feature.melspectrogram(y=y_clean, sr=sr, n_mels=128, fmax=fmax)
        spec_clean_db = librosa.power_to_db(spec_clean, ref=np.max)
        librosa.display.specshow(spec_clean_db, sr=sr, x_axis='time', y_axis='mel', 
                               ax=axes[1, 0], cmap='viridis', fmax=fmax)
        axes[1, 0].set_title(f'Mel-spectrogram\nПОСЛЕ')
        
        # 2. STFT с fmax
        n_fft = 2048
        hop_length = 512
        
        # STFT ДО
        D_orig = librosa.stft(y_orig, n_fft=n_fft, hop_length=hop_length)
        D_orig_db = librosa.amplitude_to_db(np.abs(D_orig), ref=np.max)
        librosa.display.specshow(D_orig_db, sr=sr, hop_length=hop_length,
                               x_axis='time', y_axis='hz', ax=axes[0, 1], 
                               cmap='plasma', fmax=fmax)
        axes[0, 1].set_title(f'STFT\nДО')
        
        # STFT ПОСЛЕ
        D_clean = librosa.stft(y_clean, n_fft=n_fft, hop_length=hop_length)
        D_clean_db = librosa.amplitude_to_db(np.abs(D_clean), ref=np.max)
        img = librosa.display.specshow(D_clean_db, sr=sr, hop_length=hop_length,
                                     x_axis='time', y_axis='hz', ax=axes[1, 1], 
                                     cmap='plasma', fmax=fmax)
        axes[1, 1].set_title(f'STFT\nПОСЛЕ')
        
        # 3. Chromagram
        chroma_orig = librosa.feature.chroma_stft(y=y_orig, sr=sr)
        librosa.display.specshow(chroma_orig, sr=sr, x_axis='time', y_axis='chroma',
                               ax=axes[0, 2], cmap='coolwarm')
        axes[0, 2].set_title('Chromagram\nДО')
        
        chroma_clean = librosa.feature.chroma_stft(y=y_clean, sr=sr)
        librosa.display.specshow(chroma_clean, sr=sr, x_axis='time', y_axis='chroma',
                               ax=axes[1, 2], cmap='coolwarm')
        axes[1, 2].set_title('Chromagram\nПОСЛЕ')
        
        # Добавляем информацию о классе
        for i in range(2):
            for j in range(3):
                axes[i, j].text(0.02, 0.98, f'Класс: {"КИТ" if label == 1 else "НЕТ КИТА"}', 
                              transform=axes[i, j].transAxes, fontsize=9,
                              verticalalignment='top', 
                              bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        plt.colorbar(img, ax=axes[1, 1], fraction=0.046, pad=0.04)
        plt.suptitle(f'Сравнение спектрограмм: {filename}\nЧастотный диапазон: 0-{fmax} Hz', 
                     fontsize=16, y=1.02)
        plt.tight_layout()
        plt.show()
        
        return fig
        
    except Exception as e:
        print(f"Ошибка при обработке файла {audio_file}: {e}")
        return None


In [ ]:
def extract_spectral_features(y, sr, n_mels=128, n_mfcc=13, fmax=2000):
    """
    Извлечение ключевых признаков из различных спектрограмм
    """
    features = {}
    
    try:
        # 1. MEL-СПЕКТРОГРАММА
        mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels, fmax=fmax)
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
        
        # Признаки из Mel-спектрограммы
        features['mel_spectral_centroid'] = np.mean(librosa.feature.spectral_centroid(S=mel_spec))
        features['mel_spectral_bandwidth'] = np.mean(librosa.feature.spectral_bandwidth(S=mel_spec))
        features['mel_spectral_rolloff'] = np.mean(librosa.feature.spectral_rolloff(S=mel_spec))
        features['mel_spectral_flatness'] = np.mean(librosa.feature.spectral_flatness(S=mel_spec))
        
        # Статистики по Mel-спектрограмме
        features['mel_mean'] = np.mean(mel_spec_db)
        features['mel_std'] = np.std(mel_spec_db)
        features['mel_max'] = np.max(mel_spec_db)
        
        # 2. STFT-СПЕКТРОГРАММА
        stft = np.abs(librosa.stft(y, n_fft=2048, hop_length=512))
        stft_db = librosa.amplitude_to_db(stft, ref=np.max)
        
        # Признаки из STFT
        features['stft_spectral_centroid'] = np.mean(librosa.feature.spectral_centroid(S=stft))
        features['stft_spectral_rolloff'] = np.mean(librosa.feature.spectral_rolloff(S=stft))
        features['stft_energy'] = np.sum(stft ** 2) / len(stft.flatten())
        
        # Статистики по STFT
        features['stft_mean'] = np.mean(stft_db)
        features['stft_std'] = np.std(stft_db)
        
        # 3. MFCC
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
        
        # Статистики по MFCC (первые 5 коэффициентов самые важные)
        for i in range(min(8, n_mfcc)):
            features[f'mfcc_{i}_mean'] = np.mean(mfccs[i])
            features[f'mfcc_{i}_std'] = np.std(mfccs[i])
        
        # 4. CHROMA-ПРИЗНАКИ
        chroma_stft = librosa.feature.chroma_stft(y=y, sr=sr)
        features['chroma_stft_mean'] = np.mean(chroma_stft)
        features['chroma_stft_std'] = np.std(chroma_stft)
        
        # 5. SPECTRAL CONTRAST
        spectral_contrast = librosa.feature.spectral_contrast(y=y, sr=sr, fmin=50.0)
        features['spectral_contrast_mean'] = np.mean(spectral_contrast)
        
        # 6. TEMPORAL FEATURES
        features['zcr_mean'] = np.mean(librosa.feature.zero_crossing_rate(y))
        features['rms_mean'] = np.mean(librosa.feature.rms(y=y))
        
        # 7. LOW-FREQUENCY ENERGY
        freqs = librosa.fft_frequencies(sr=sr, n_fft=2048)
        low_freq_mask = (freqs >= 10) & (freqs <= 500)
        if np.any(low_freq_mask):
            low_freq_energy = np.mean(stft_db[low_freq_mask, :])
            features['low_freq_energy'] = low_freq_energy
        else:
            features['low_freq_energy'] = 0
            
    except Exception as e:
        print(f"Ошибка при извлечении признаков: {e}")
    
    return features

In [ ]:
def create_spectral_features_dataset(audio_files, labels_df, sr=22050, use_denoising=True):
    """
    Создание датасета с признаками из спектрограмм
    """
    features_list = []
    labels_list = []
    filenames_list = []
    
    for i, audio_file in enumerate(audio_files):
        try:
            if i % 100 == 0:
                print(f"Обработано {i}/{len(audio_files)} файлов...")
            
            # Загрузка аудио
            y, sr = librosa.load(audio_file, sr=sr, duration=2.0)
            
            # Шумоподавление
            if use_denoising:
                y_processed = whale_denoise(y, sr)
            else:
                y_processed = y
            
            # Извлечение признаков
            features = extract_spectral_features(y_processed, sr)
            
            # Получение метки
            filename = audio_file.name
            matching_row = labels_df[labels_df['clip_name'] == filename]
            
            if not matching_row.empty:
                label = matching_row['label'].iloc[0]
                
                features_list.append(features)
                labels_list.append(label)
                filenames_list.append(filename)
            
        except Exception as e:
            print(f"Ошибка обработки {audio_file}: {e}")
            continue
    
    # Преобразование в DataFrame
    features_df = pd.DataFrame(features_list)
    features_df['label'] = labels_list
    features_df['filename'] = filenames_list
    
    print(f"\nУспешно обработано: {len(features_df)} файлов")
    print(f"Размерность признаков: {features_df.shape[1] - 2} признаков")
    
    return features_df

In [ ]:
def analyze_feature_importance(features_df, top_k=20):
    """
    Анализ важности признаков
    """
    # Подготовка данных
    X = features_df.drop(['label', 'filename'], axis=1, errors='ignore')
    y = features_df['label']
    
    # Заполнение пропущенных значений
    X = X.fillna(X.mean())
    
    # Масштабирование
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Обучение Random Forest для анализа важности
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_scaled, y)
    
    # Важность признаков
    feature_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': rf.feature_importances_
    }).sort_values('importance', ascending=False)
    
    # Визуализация
    plt.figure(figsize=(12, 8))
    top_features = feature_importance.head(top_k)
    
    plt.barh(range(len(top_features)), top_features['importance'])
    plt.yticks(range(len(top_features)), top_features['feature'])
    plt.xlabel('Важность признака')
    plt.title(f'Топ-{top_k} самых важных признаков')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    print("Топ-10 самых важных признаков:")
    for i, row in top_features.head(10).iterrows():
        print(f"{row['feature']}: {row['importance']:.4f}")
    
    return feature_importance, top_features

In [ ]:
import os
from pathlib import Path
#Файлы
large_data_path = r'F:\datasets\whale-detection-challenge\whale_data\data' 
train_path = Path(large_data_path) / 'train'
test_path = Path(large_data_path) / 'test'

#Метки
train_csv_path = Path(large_data_path) / 'train.csv'
train_df = pd.read_csv(train_csv_path)

train_df = train_df.rename(columns={'clip_name_label': 'filename', 'train1.atif,0': 'label'})

train_audio_files = list(train_path.rglob('*.aiff')) + list(train_path.rglob('*.aif'))
test_audio_files = list(test_path.rglob('*.aiff')) + list(test_path.rglob('*.aif'))

print(f"Размер датасета: {len(train_df)}")
print(f"Распределение меток:\n{train_df['label'].value_counts()}")

In [ ]:
 # Сначала протестируем на маленькой выборке
sample_files = train_audio_files[:500]  # первые 500 файлов для теста
    
print("🚀 ЗАПУСК ОБУЧЕНИЯ НА ПРИЗНАКАХ ИЗ СПЕКТРОГРАММ")
X_train, X_test, y_train, y_test, features_df, importance = train_on_spectral_features(
    sample_files, train_df, use_top_features=True
)
    
# Обучение моделей
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss'),
    'SVM': SVC(kernel='rbf', random_state=42, probability=True),
    'Logistic Regression': LogisticRegression(random_state=42)
}
    
print("\n📊 СРАВНЕНИЕ МОДЕЛЕЙ:")
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    results[name] = accuracy
    print(f"{name}: {accuracy:.4f}")
    
# Лучшая модель
best_model_name = max(results, key=results.get)
best_model = models[best_model_name]
y_pred_best = best_model.predict(X_test)
    
print(f"\n🎯 ЛУЧШАЯ МОДЕЛЬ: {best_model_name}")
print("📈 ОТЧЕТ ПО КЛАССИФИКАЦИИ:")
print(classification_report(y_test, y_pred_best))

In [ ]:
# ВИЗУАЛИЗАЦИЯ СПЕКТРОГРАММ ДО И ПОСЛЕ ШУМОПОДАВЛЕНИЯ
print("🔊 ВИЗУАЛИЗАЦИЯ СПЕКТРОГРАММ")

# Находим по одному файлу каждого класса для демонстрации
class_0_sample = None
class_1_sample = None

for audio_file in train_audio_files[:200]:
    filename = audio_file.name
    matching_row = train_df[train_df['clip_name'] == filename]
    if not matching_row.empty:
        label = matching_row['label'].iloc[0]
        if label == 0 and class_0_sample is None:
            class_0_sample = audio_file
        elif label == 1 and class_1_sample is None:
            class_1_sample = audio_file
        
        if class_0_sample and class_1_sample:
            break
# Визуализируем спектрограммы для каждого класса
if class_0_sample:
    print(f"\n📊 Файл БЕЗ кита: {class_0_sample.name}")
    compare_spectrograms(class_0_sample, label=0)

if class_1_sample:
    print(f"\n📊 Файл С китом: {class_1_sample.name}")
    compare_spectrograms(class_1_sample, label=1)


In [ ]:
print("\n📈 ВИЗУАЛИЗАЦИЯ РАСПРЕДЕЛЕНИЯ КЛАССОВ НА ПЛОСКОСТИ")

def visualize_class_distribution_simple(X, y, title="Распределение классов"):
    """
    Упрощенная визуализация распределения классов в 2D пространстве
    """
    from sklearn.decomposition import PCA
    from sklearn.manifold import TSNE
    from sklearn.preprocessing import StandardScaler
    
    # Масштабируем признаки
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Создаем фигуру
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # 1. PCA проекция
    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X_scaled)
    
    # Цвета для классов
    colors = ['red', 'blue']
    labels = ['Нет кита', 'Кит']
    
    for i in [0, 1]:
        mask = (y == i)
        ax1.scatter(X_pca[mask, 0], X_pca[mask, 1], 
                   c=colors[i], alpha=0.6, label=labels[i], s=30)
    
    ax1.set_title(f'PCA проекция\nОбъясненная дисперсия: {pca.explained_variance_ratio_.sum():.3f}')
    ax1.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%})')
    ax1.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%})')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. t-SNE проекция
    try:
        tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(X_scaled)-1))
        X_tsne = tsne.fit_transform(X_scaled)
        
        for i in [0, 1]:
            mask = (y == i)
            ax2.scatter(X_tsne[mask, 0], X_tsne[mask, 1], 
                       c=colors[i], alpha=0.6, label=labels[i], s=30)
        
        ax2.set_title('t-SNE проекция')
        ax2.set_xlabel('t-SNE 1')
        ax2.set_ylabel('t-SNE 2')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
    except Exception as e:
        print(f"t-SNE error: {e}")
        ax2.text(0.5, 0.5, 't-SNE failed', ha='center', va='center')
    
    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()
    
    # Анализ разделимости
    mean_pc1_class0 = np.mean(X_pca[y == 0, 0])
    mean_pc1_class1 = np.mean(X_pca[y == 1, 0])
    pc1_separation = abs(mean_pc1_class1 - mean_pc1_class0)
    
    print(f"🔍 АНАЛИЗ РАЗДЕЛИМОСТИ:")
    print(f"Разделение по PC1: {pc1_separation:.3f}")
    
    if pc1_separation > 1.0:
        print("✅ Классы хорошо разделяются линейно")
        print("   Рекомендуются: Logistic Regression, Linear SVM")
    elif pc1_separation > 0.5:
        print("⚠️  Классы умеренно разделяются") 
        print("   Рекомендуются: Random Forest, XGBoost, Kernel SVM")
    else:
        print("❌ Классы плохо разделяются")
        print("   Рекомендуются: Сложные модели (XGBoost, Neural Networks)")
    
    return pc1_separation

# Визуализируем распределение классов на тренировочных данных
print("\n📊 РАСПРЕДЕЛЕНИЕ КЛАССОВ НА ТРЕНИРОВОЧНЫХ ДАННЫХ")
pc1_sep = visualize_class_distribution_simple(X_train, y_train, 
                                            "Распределение классов в тренировочных данных")

# Также визуализируем на тестовых данных
print("\n📊 РАСПРЕДЕЛЕНИЕ КЛАССОВ НА ТЕСТОВЫХ ДАННЫХ")
visualize_class_distribution_simple(X_test, y_test, 
                                  "Распределение классов в тестовых данных")

# ДОПОЛНИТЕЛЬНО: ВИЗУАЛИЗАЦИЯ ВАЖНОСТИ ПРИЗНАКОВ
print("\n📋 ВИЗУАЛИЗАЦИЯ ВАЖНОСТИ ПРИЗНАКОВ")

def plot_feature_importance_basic(importance_df, top_n=15):
    """
    Базовая визуализация важности признаков
    """
    top_features = importance_df.head(top_n)
    
    plt.figure(figsize=(12, 8))
    plt.barh(range(len(top_features)), top_features['importance'])
    plt.yticks(range(len(top_features)), top_features['feature'])
    plt.xlabel('Важность признака')
    plt.title(f'Топ-{top_n} самых важных признаков для обнаружения китов')
    plt.gca().invert_yaxis()
    plt.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()
    
    print("🎯 Топ-5 самых важных признаков:")
    for i, row in top_features.head(5).iterrows():
        print(f"   {row['feature']}: {row['importance']:.4f}")

# Визуализируем важность признаков
plot_feature_importance_basic(importance)

# ИТОГОВЫЙ АНАЛИЗ
print("\n" + "="*50)
print("🎯 ИТОГОВЫЙ АНАЛИЗ ДАННЫХ")
print("="*50)

print(f"📊 Общая информация:")
print(f"   • Размер тренировочных данных: {X_train.shape}")
print(f"   • Размер тестовых данных: {X_test.shape}")
print(f"   • Количество признаков: {X_train.shape[1]}")
print(f"   • Разделение по PC1: {pc1_sep:.3f}")

print(f"\n📈 Результаты моделей:")
for name, accuracy in results.items():
    print(f"   • {name}: {accuracy:.4f}")

print(f"\n🏆 Лучшая модель: {best_model_name} ({results[best_model_name]:.4f})")

# Анализ баланса классов
unique, counts = np.unique(y_train, return_counts=True)
print(f"\n⚖️  Баланс классов в тренировочных данных:")
for cls, count in zip(unique, counts):
    percentage = count / len(y_train) * 100
    label = "КИТ" if cls == 1 else "НЕТ КИТА"
    print(f"   • {label}: {count} samples ({percentage:.1f}%)")

print("\n🐋 РЕКОМЕНДАЦИИ:")
if pc1_sep > 1.0 and results[best_model_name] > 0.85:
    print("✅ Отличные результаты! Модель хорошо справляется с обнаружением китов.")
elif pc1_sep > 0.5 and results[best_model_name] > 0.75:
    print("⚠️  Хорошие результаты. Можно попробовать улучшить через:")
    print("   - Больше данных для обучения")
    print("   - Дополнительные признаки из спектрограмм")
    print("   - Настройку гиперпараметров моделей")
else:
    print("❌ Результаты можно улучшить. Рекомендуется:")
    print("   - Увеличить объем данных")
    print("   - Попробовать другие методы шумоподавления")
    print("   - Использовать нейросетевые подходы")

In [ ]:
print("🔍 АНАЛИЗ ПРОБЛЕМЫ: КЛАССЫ НАКЛАДЫВАЮТСЯ")
print("=" * 50)

# Более детальный анализ перекрытия классов
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import numpy as np

# Масштабируем данные
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)

# PCA анализ
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Анализ перекрытия в PCA пространстве
def analyze_class_overlap(X_pca, y):
    print("📊 АНАЛИЗ ПЕРЕКРЫТИЯ КЛАССОВ:")
    
    # Разделяем данные по классам
    class_0 = X_pca[y == 0]
    class_1 = X_pca[y == 1]
    
    # Вычисляем средние и стандартные отклонения
    mean_0 = np.mean(class_0, axis=0)
    mean_1 = np.mean(class_1, axis=0)
    std_0 = np.std(class_0, axis=0)
    std_1 = np.std(class_1, axis=0)
    
    # Расстояние между центрами кластеров
    distance = np.linalg.norm(mean_1 - mean_0)
    print(f"• Расстояние между центрами кластеров: {distance:.3f}")
    
    # Перекрытие по PC1
    min_0_pc1, max_0_pc1 = np.min(class_0[:, 0]), np.max(class_0[:, 0])
    min_1_pc1, max_1_pc1 = np.min(class_1[:, 0]), np.max(class_1[:, 0])
    
    overlap_pc1 = max(0, min(max_0_pc1, max_1_pc1) - max(min_0_pc1, min_1_pc1))
    total_range_pc1 = max(max_0_pc1, max_1_pc1) - min(min_0_pc1, min_1_pc1)
    overlap_percentage_pc1 = (overlap_pc1 / total_range_pc1) * 100 if total_range_pc1 > 0 else 0
    
    print(f"• Перекрытие по PC1: {overlap_percentage_pc1:.1f}%")
    
    # Анализ плотности в области перекрытия
    overlap_region_min = max(min_0_pc1, min_1_pc1)
    overlap_region_max = min(max_0_pc1, max_1_pc1)
    
    points_in_overlap_0 = np.sum((class_0[:, 0] >= overlap_region_min) & (class_0[:, 0] <= overlap_region_max))
    points_in_overlap_1 = np.sum((class_1[:, 0] >= overlap_region_min) & (class_1[:, 0] <= overlap_region_max))
    
    total_overlap = points_in_overlap_0 + points_in_overlap_1
    overlap_percentage = (total_overlap / len(X_pca)) * 100
    
    print(f"• Точек в области перекрытия: {total_overlap} ({overlap_percentage:.1f}% всех данных)")
    
    return distance, overlap_percentage

distance, overlap_percentage = analyze_class_overlap(X_pca, y_train)

print(f"\n🎯 ВЫВОДЫ:")
if distance < 1.0:
    print("• Классы СИЛЬНО перекрываются - это сложная задача классификации")
elif distance < 2.0:
    print("• Классы УМЕРЕННО перекрываются - нужны хорошие алгоритмы")
else:
    print("• Классы хорошо разделены")

if overlap_percentage > 30:
    print("• Большая область перекрытия - высокий уровень шума в данных")